# Lab 1 — First Steps into Agentic AI

My first practical lab working with LLM APIs and multi-step AI workflows.

**Focus:** API setup, prompting, chaining LLM calls, and evaluating responses.

*Based on concepts from Ed Donner's Agentic AI course, with my own adaptations and experiments.*


## 1. Setting up the API environment

I’m using environment variables to keep API credentials separate from the notebook code.


In [27]:
# Load the environment-variable helper.


from dotenv import load_dotenv


In [28]:
# Load the variables from .env so the API credentials are available to Python.


load_dotenv(override=True)



True

In [29]:
# Quick check that the notebook is running.

print("TEST")


TEST


The API key is loaded from `.env` rather than written directly into the notebook.

Only a short prefix is displayed when checking the key, so the full credential is not exposed.


In [30]:

import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("OpenAI API Key is available.")
else:
    print("OpenAI API Key not set.")
    


OpenAI API Key is available.


## 2. Making the first LLM call

I’m using the OpenAI client to send a simple agriculture-related prompt and inspect the response.


In [31]:
# Import the OpenAI client used for the API calls in this lab.


from openai import OpenAI


In [32]:
# Create a client using the API credentials loaded from the environment.


openai = OpenAI()


In [45]:
# Start with a simple agriculture-focused prompt.


messages = [{"role": "user", "content": "tell me a fun fact in agriculture"}]


In [46]:
messages

[{'role': 'user', 'content': 'tell me a fun fact in agriculture'}]

In [47]:
# Send the prompt to the model and display its response.


response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)


Fun fact: **Bananas are botanically classified as berries**—and many bananas that end up in grocery stores are seedless because of how they’re cultivated.


## 3. Chaining LLM calls

Next, I use one LLM call to generate a challenging question, then use another call to answer it.

This is where the workflow starts becoming more interesting: the output from one step becomes the input to the next.


In [48]:
# Ask the model to generate a challenging question.


question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]


In [37]:
# Generate the question with the selected model. "gpt-5.4-mini"


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
print(question)


A prison has 100 cells in a row, all initially closed, and 100 prisoners enter one at a time. Prisoner 1 toggles every cell, prisoner 2 toggles every 2nd cell, prisoner 3 toggles every 3rd cell, and so on until prisoner 100 toggles only cell 100. After all prisoners have passed, which cells are open, and why?


In [38]:
# Pass the generated question into the next step.


messages = [{"role": "user", "content": question}]
messages


[{'role': 'user',
  'content': 'A prison has 100 cells in a row, all initially closed, and 100 prisoners enter one at a time. Prisoner 1 toggles every cell, prisoner 2 toggles every 2nd cell, prisoner 3 toggles every 3rd cell, and so on until prisoner 100 toggles only cell 100. After all prisoners have passed, which cells are open, and why?'}]

In [39]:
# Ask the model to answer the generated question.


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
answer = response.choices[0].message.content
print(answer)


The **open cells are exactly the ones with perfect-square numbers**:

**1, 4, 9, 16, 25, 36, 49, 64, 81, 100**

### Why?
A cell is toggled once for every prisoner number that divides that cell number.

- Example: cell 12 is toggled by prisoners 1, 2, 3, 4, 6, 12
- So it gets toggled **6 times**.
- Since it starts closed, an **even** number of toggles leaves it closed, and an **odd** number leaves it open.

Most numbers have divisors in pairs:
- If \( d \) divides \( n \), then \( \frac{n}{d} \) also divides \( n \)
- So divisors usually come in pairs, giving an even number of toggles

But **perfect squares** have one divisor that pairs with itself:
- For example, 36 has divisor 6 paired with itself because \(6 \times 6 = 36\)
- That means perfect squares have an **odd** number of divisors

So only the perfect-square cells are toggled an odd number of times, and thus remain open.


In [40]:
from IPython.display import Markdown, display

display(Markdown(answer))

The **open cells are exactly the ones with perfect-square numbers**:

**1, 4, 9, 16, 25, 36, 49, 64, 81, 100**

### Why?
A cell is toggled once for every prisoner number that divides that cell number.

- Example: cell 12 is toggled by prisoners 1, 2, 3, 4, 6, 12
- So it gets toggled **6 times**.
- Since it starts closed, an **even** number of toggles leaves it closed, and an **odd** number leaves it open.

Most numbers have divisors in pairs:
- If \( d \) divides \( n \), then \( \frac{n}{d} \) also divides \( n \)
- So divisors usually come in pairs, giving an even number of toggles

But **perfect squares** have one divisor that pairs with itself:
- For example, 36 has divisor 6 paired with itself because \(6 \times 6 = 36\)
- That means perfect squares have an **odd** number of divisors

So only the perfect-square cells are toggled an odd number of times, and thus remain open.

## 4. Evaluating the response

I then pass both the question and answer to another LLM call and ask it to judge whether the answer is correct.

This gives the workflow a simple **generate → answer → evaluate** pattern.


In [41]:
# Combine the question and answer into an evaluation prompt.

message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(message)



Here is a question:
A prison has 100 cells in a row, all initially closed, and 100 prisoners enter one at a time. Prisoner 1 toggles every cell, prisoner 2 toggles every 2nd cell, prisoner 3 toggles every 3rd cell, and so on until prisoner 100 toggles only cell 100. After all prisoners have passed, which cells are open, and why?

And here is a possible answer that might be correct or incorrect:
The **open cells are exactly the ones with perfect-square numbers**:

**1, 4, 9, 16, 25, 36, 49, 64, 81, 100**

### Why?
A cell is toggled once for every prisoner number that divides that cell number.

- Example: cell 12 is toggled by prisoners 1, 2, 3, 4, 6, 12
- So it gets toggled **6 times**.
- Since it starts closed, an **even** number of toggles leaves it closed, and an **odd** number leaves it open.

Most numbers have divisors in pairs:
- If \( d \) divides \( n \), then \( \frac{n}{d} \) also divides \( n \)
- So divisors usually come in pairs, giving an even number of toggles

But **per

In [42]:
# Ask another model to evaluate the answer.

messages = [{"role": "user", "content": message}]
response = openai.chat.completions.create(model="gpt-5.4", messages=messages)
print(response.choices[0].message.content)


Correct. The answer is accurate.

- A cell numbered \(n\) is toggled once for each divisor of \(n\).
- So the total number of toggles equals the number of divisors of \(n\).
- A cell ends open iff it is toggled an odd number of times.
- Numbers usually have divisors in pairs \((d, n/d)\), so they have an even number of divisors.
- Perfect squares are the exception, because one divisor is repeated in the middle, e.g. for 36, \(6 \times 6 = 36\), so they have an odd number of divisors.

Therefore the open cells are exactly the perfect squares from 1 to 100:

**1, 4, 9, 16, 25, 36, 49, 64, 81, 100**.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>
        </td>
    </tr>
</table>

## 5. My agriculture experiment

I applied the same idea to an agriculture use case: identify a pain point and ask the model to propose an Agentic AI solution.

This helped me connect the technical exercise to a domain I already understand.


In [ ]:
# Identify an agriculture pain point that could be explored by an AI agent.


messages = [{"role": "user", "content": "what is one pain-point in agriculture that is worth exploring by an agent. In plain text and just the pain-point"}]


response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
print(question)


One strong pain-point worth exploring with an agent is **early detection of crop disease or pest outbreaks**.

Why this is a good target:
- Farmers often notice problems **too late**, after damage has already spread.
- It requires **continuous monitoring** of fields, weather, and plant health data.
- An agent could help by **analyzing images, sensor data, and forecasts**, then alerting the farmer with likely causes and next steps.
- It has clear value because catching issues early can **save yield, reduce pesticide use, and lower costs**.


In [44]:
# Turn the identified pain point into a proposed Agentic AI solution.

messages = [{
    "role": "user",
    "content": f"""
Here is a pain-point in agriculture:
{question}

Propose an Agentic AI solution for this pain-point.
Briefly explain what the agent would do, which tools it might use, and how it would help.
Respond in plain text only.
Do not use markdown.
"""
}]

response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
solution = response.choices[0].message.content
print(solution)



An agentic AI solution could be a Crop Health Monitoring Agent that continuously watches for signs of disease or pest outbreaks and alerts the farmer before the problem spreads.

What the agent would do:
It would collect and review data from multiple sources, such as field images from drones or smartphones, soil and weather sensors, satellite imagery, and local forecast data. It would look for unusual patterns like leaf discoloration, wilting, canopy changes, pest damage, or weather conditions that favor outbreaks. If it detects a likely issue, it would rank the most probable causes and send an alert with recommended next steps, such as scouting a specific area, applying treatment, or adjusting irrigation.

Tools it might use:
- Computer vision models to analyze plant images for disease symptoms or insect damage
- Sensor data ingestion from IoT devices measuring temperature, humidity, soil moisture, and leaf wetness
- Weather and forecast APIs to predict conditions that increase diseas

## Key Takeaway

This lab introduced me to the idea of connecting multiple LLM calls into a workflow.

I also started experimenting with how these patterns could be applied to agriculture, something I’ll build on in later labs.
